# Inspecting a capture

`scripts/capture.py` stores, for every requested layer, the sparse SAE code of every token: which features fire and
how strongly. This notebook shows how to read that output and look at it: one prompt token by token, the features a
prompt is made of, where a feature fires, which features belong to a domain, and how much two domains share.

Make a small capture first. Gemma 3 270M, three domains, 50 prompts each, every layer, takes a few minutes on one GPU:

```
python scripts/capture.py --model 270m --prompts thesis/data/prompts.csv \
    --domains par_lang_english,par_lang_finnish,div_code_python --n-per-domain 50 --out runs/example
```

Any prompt CSV with the columns `prompt_id, micro_domain, text` works. The notebook needs only numpy, pandas,
matplotlib and the tokenizer from `transformers`; no GPU.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # run from notebooks/; the repository root holds saepipe/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import HTML, display
from saepipe.reader import Capture

CAPTURE = os.environ.get("CAPTURE", "../runs/example")
cap = Capture(CAPTURE)
print(cap.n_prompts, "prompts,", int(cap.n_tokens.sum()), "tokens, layers", cap.layers[0], "to", cap.layers[-1])
cap.prompts.groupby("domain").agg(prompts=("prompt_id", "size"), mean_tokens=("n_tokens", "mean"))

## 1. One prompt, token by token

`cap.codes(L, p)` returns the firings of prompt `p` at layer `L`: the position of each firing in the prompt, the
feature id and its activation. Position 0 is `<bos>`; the capture does not encode it by default.

In [ ]:
L = cap.layers[len(cap.layers) // 2]                # a middle layer
p = 0
toks = cap.tokens(p); pos, feat, val = cap.codes(L, p)
per_token = pd.DataFrame({"token": toks, "features firing": np.bincount(pos, minlength=len(toks))})
strongest = pd.DataFrame({"pos": pos, "feat": feat, "act": val}).sort_values("act", ascending=False).groupby("pos").head(3)
per_token["strongest three (feature: activation)"] = [
    ", ".join(f"{r.feat}: {r.act:.0f}" for r in strongest[strongest.pos == i].itertuples()) for i in range(len(toks))]
print(f"layer {L}, prompt {cap.prompt_ids[p]} ({cap.domains[p]})")
per_token.head(20)

## 2. What a prompt is made of

A feature's weight in a prompt: its activation summed over the tokens, divided by the number of tokens. Gemma Scope
decoder directions have unit norm, so this is how much the feature writes into the residual stream per token.

In [ ]:
n = len(toks) - 1                                   # tokens after <bos>
w = pd.DataFrame({"feat": feat, "act": val}).groupby("feat").agg(weight=("act", "sum"), tokens=("act", "size"))
w["weight"] /= n; w["share of tokens"] = w["tokens"] / n
w.sort_values("weight", ascending=False).head(15)

## 3. Where a feature fires in a text

In [ ]:
def show(L, p, f, pad=None):
    toks = cap.tokens(p); pos, feat, val = cap.codes(L, p); a = dict(zip(pos[feat == f].tolist(), val[feat == f].tolist()))
    if not a:
        return HTML(f"<p>feature {f} does not fire in prompt {p} at layer {L}</p>")
    lo, hi = (0, len(toks)) if pad is None else (max(1, min(a) - pad), min(len(toks), max(a) + pad + 1))
    top = max(a.values()); out = []
    for i in range(max(lo, 1), hi):
        t = toks[i].replace("<", "&lt;")
        out.append(f'<mark style="background:rgba(255,170,30,{0.2 + 0.8 * a[i] / top:.2f})" title="{a[i]:.1f}">{t}</mark>' if i in a else t)
    head = f"<b>layer {L} · feature {f} · prompt {cap.prompt_ids[p]} ({cap.domains[p]}) · {len(a)} of {len(toks) - 1} tokens</b><br>"
    return HTML(head + '<div style="font-family:monospace;white-space:pre-wrap;line-height:1.6">' + "".join(out) + "</div>")

f_top = int(w.sort_values("weight", ascending=False).index[0])
show(L, p, f_top)

## 4. Features of a domain

`cap.firing_rate(L)` gives, for every domain and feature, the share of the domain's prompts in which the feature
fires (on any token after `<bos>`). Features that fire in most prompts of every domain are always on and say little
about any one domain; a feature that fires in most prompts of one domain and rarely elsewhere is specific to it.

In [ ]:
names, rate = cap.firing_rate(L)
corpus = (rate * cap.prompts.groupby("domain").size().loc[names].values[:, None]).sum(0) / cap.n_prompts
always_on = corpus >= 0.6
rows = []
for i, d in enumerate(names):
    others = np.delete(rate, i, axis=0).max(0)
    specific = np.where((rate[i] >= 0.5) & (others <= 0.05))[0]
    rows.append({"domain": d, "features firing": int((rate[i] > 0).sum()), "in half its prompts": int((rate[i] >= 0.5).sum()),
                 "specific to it": len(specific), "examples": ", ".join(map(str, specific[np.argsort(-rate[i, specific])][:5]))})
print(f"layer {L}: {int(always_on.sum())} features fire in at least 60 % of all prompts")
pd.DataFrame(rows)

## 5. How much two domains share, layer by layer

The feature set of a domain: the features that fire in at least 10 % of its prompts, always-on features removed.
The Jaccard index of two sets is the size of their intersection over the size of their union. This reads every layer
of the capture once.

In [ ]:
MIN_RATE = 0.10
COMPARE = names[:4]                                 # the domains to compare; every pair of them is drawn
pairs = [(a, b) for i, a in enumerate(COMPARE) for b in COMPARE[i + 1:]]
J = {pr: [] for pr in pairs}
for l in cap.layers:
    nm, r = cap.firing_rate(l)
    corpus_l = (r * cap.prompts.groupby("domain").size().loc[nm].values[:, None]).sum(0) / cap.n_prompts
    sets = {d: set(np.where((r[i] >= MIN_RATE) & (corpus_l < 0.6))[0]) for i, d in enumerate(nm) if d in COMPARE}
    for a, b in pairs:
        J[(a, b)].append(len(sets[a] & sets[b]) / max(len(sets[a] | sets[b]), 1))
fig, ax = plt.subplots(figsize=(7, 3.5))
for (a, b), y in J.items():
    ax.plot(cap.layers, y, label=f"{a} · {b}")
ax.set_xlabel("layer"); ax.set_ylabel("Jaccard index of the feature sets"); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.show()

## 6. Where one feature fires across the capture

In [ ]:
def where(L, f, n_show=3, pad=12):
    c = cap.layer(L); m = (c["feat"] == f) & (c["pos"] > 0)
    hits = pd.DataFrame({"prompt": c["prompt"][m], "act": c["val"][m]}).groupby("prompt").agg(tokens=("act", "size"), peak=("act", "max"))
    hits["domain"] = cap.domains[hits.index]
    display(hits.groupby("domain").size().rename("prompts it fires in").to_frame())
    for q in hits.sort_values("tokens", ascending=False).index[:n_show]:
        display(show(L, int(q), f, pad=pad))

where(L, f_top)

## Further

For statistics over many prompts and every layer, `scripts/aggregate.py` reduces a capture to per-domain counts and
activation sums per feature, one file per layer. The decoder direction of a feature, to steer with it or to compare
features, is row `f` of `W_dec` in the Gemma Scope 2 file of that layer.